In [2]:
import requests
import pandas as pd

from datetime import datetime

from openai import OpenAI
from bs4 import BeautifulSoup

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key="<YOUR_GITHUB_TOKEN>"
)

In [3]:
player_name = 'Raphinha'
url = 'https://fbref.com/en/players/3423f250/Raphinha'
attrs = 'scout_summary_AM'

response = requests.get(url, verify=False)
df = pd.read_html(response.text, attrs={'id': attrs})[0]

/Users/miwebo/Desktop/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1013: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fbref.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  warnings.warn(
/var/folders/2l/98dn0dvj6jx4l166nprz2mc00000gn/T/ipykernel_47438/1811154900.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(response.text, attrs={'id': attrs})[0]


In [4]:
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [5]:
# Get the player's age, team, league, and position
position = soup.select_one('p:-soup-contains("Position")').text.split(':')[-2].split(' ')[0].strip()
birthday = soup.select_one('span[id="necro-birth"]').text.strip()

# get the player's age
age = (datetime.now() - datetime.strptime(birthday, '%B %d, %Y')).days // 365
team = soup.select_one('p:-soup-contains("Club")').text.split(':')[-1].strip()

In [6]:
# df.columns = df.columns.droplevel(0)

df = df.dropna(subset='Statistic')

In [7]:
prompt = f"""

I need you to create a scouting report on {player_name}. Can you provide me with a summary of their strengths and weaknesses?

Here is the data I have on him:

Player: {player_name}
Position: {position}
Age: {age}
Team: {team}

{df.to_markdown()}


Return the scouting report in the following markdown format:

# Scouting Report for {player_name}

## Strengths
< a list of 1 to 3 strengths >

## Weaknesses
< a list of 1 to 3 weaknesses >

## Summary
< a brief summary of the player's overall performance and if he would be beneficial to the team >
"""

In [8]:
print(prompt)



I need you to create a scouting report on Raphinha. Can you provide me with a summary of their strengths and weaknesses?

Here is the data I have on him:

Player: Raphinha
Position: FW-MF (AM-WM)
Age: 28
Team: Barcelona

|    | Statistic                | Per 90   |   Percentile |
|---:|:-------------------------|:---------|-------------:|
|  0 | Non-Penalty Goals        | 0.66     |           98 |
|  1 | npxG: Non-Penalty xG     | 0.51     |           98 |
|  2 | Shots Total              | 3.57     |           93 |
|  3 | Assists                  | 0.38     |           94 |
|  4 | xAG: Exp. Assisted Goals | 0.43     |           99 |
|  5 | npxG + xAG               | 0.94     |           99 |
|  6 | Shot-Creating Actions    | 5.43     |           91 |
|  8 | Passes Attempted         | 44.44    |           75 |
|  9 | Pass Completion %        | 71.3%    |           19 |
| 10 | Progressive Passes       | 4.29     |           62 |
| 11 | Progressive Carries      | 3.06     |           43

In [9]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a professional football (soccer) scout.",
        },
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-4o",
    temperature=1,
    max_tokens=4096,
    top_p=1
)

In [10]:
print(response.choices[0].message.content)

# Scouting Report for Raphinha

## Strengths
1. **Elite Goal-Scoring Ability**: Raphinha excels in front of goal, evidenced by his Non-Penalty Goals (0.66 per 90, 98th percentile) and Non-Penalty xG (0.51 per 90, 98th percentile). His clinical finishing makes him a constant threat in attacking areas.
2. **Creative Playmaker**: Raphinha ranks in the 99th percentile for Expected Assisted Goals (0.43 per 90) and contributes heavily to build-up play with 5.43 Shot-Creating Actions per 90 (91st percentile). His vision and passing in the final third are world-class.
3. **Offensive Presence in Dangerous Areas**: With 5.28 touches in the attacking penalty area (73rd percentile) and consistently receiving progressive passes (9.90 per 90, 78th percentile), he positions himself effectively to create and finish scoring opportunities.

## Weaknesses
1. **Passing Accuracy**: At 71.3% (19th percentile), Raphinha's pass completion rate is notably low, potentially leading to turnovers in possession and

In [14]:
# save the scouting report to a new markdown file
with open(f"{player_name}_report.md", "w") as f:
    f.write(response.choices[0].message.content),

In [16]:
from IPython.display import Markdown, display

# Open and read the markdown file
with open(f"{player_name}_report.md", "r") as f:
    markdown_content = f.read()

# Display the markdown content
display(Markdown(markdown_content))

# Scouting Report for Raphinha

## Strengths
1. **Elite Goal-Scoring Ability**: Raphinha excels in front of goal, evidenced by his Non-Penalty Goals (0.66 per 90, 98th percentile) and Non-Penalty xG (0.51 per 90, 98th percentile). His clinical finishing makes him a constant threat in attacking areas.
2. **Creative Playmaker**: Raphinha ranks in the 99th percentile for Expected Assisted Goals (0.43 per 90) and contributes heavily to build-up play with 5.43 Shot-Creating Actions per 90 (91st percentile). His vision and passing in the final third are world-class.
3. **Offensive Presence in Dangerous Areas**: With 5.28 touches in the attacking penalty area (73rd percentile) and consistently receiving progressive passes (9.90 per 90, 78th percentile), he positions himself effectively to create and finish scoring opportunities.

## Weaknesses
1. **Passing Accuracy**: At 71.3% (19th percentile), Raphinha's pass completion rate is notably low, potentially leading to turnovers in possession and missed opportunities to sustain attacking momentum.
2. **One-on-One Dribbling**: His Successful Take-Ons (1.51 per 90, 45th percentile) could be improved for a player operating in wide and advanced positions. This indicates he may not consistently beat defenders in 1v1 situations.
3. **Defensive Contribution**: While solid in blocks (55th percentile) and clearances (63rd percentile), Raphinha lags in interceptions (33rd percentile) and aerial duels (9th percentile). His defensive work rate and ability to contest physically could be limiting for his team in high-press scenarios.

## Summary
Raphinha is a highly effective forward/winger whose primary strengths lie in his goal-scoring and creative abilities. He is one of the most dangerous attackers in Europe, combining elite finishing with exceptional playmaking. However, his relatively poor passing accuracy and mediocre success in 1v1 duels occasionally hinder his efficiency in possession, while his defensive contributions are unlikely to provide much support in pressing systems or physical battles.

Overall, Raphinha would be a tremendous asset to teams looking for a dynamic, goal-oriented winger who can create and convert chances in key moments. Adjustments to his passing game and defensive mindset could further elevate his already elite profile.